# 06 A/B Simulation - Historical Backtest

目标：用历史首单用户做一次 A/B 演练，验证 `First-order SLA Upgrade` 在一个可解释 uplift 场景下会不会改善 `30d repeat`、`late share`、`bad review share`，以及护栏是否恶化。

说明：当前环境没有 `pandas / scipy / jupyter`，因此 notebook 只使用 `sqlite3 + Python standard library`，保证可以在本项目环境里直接执行。

In [ ]:
from pathlib import Path
import hashlib
import math
import sqlite3
from collections import defaultdict
from statistics import NormalDist

ROOT = Path.cwd()
if not (ROOT / 'sql').exists():
    ROOT = ROOT.parent
DB_PATH = ROOT / 'data' / 'raw' / 'olist.sqlite'
SEED = '20260416'
SIM_DELAY_REDUCTION_DAYS = 1.5
SIM_REPEAT_UPLIFT_ABS = 0.0035
SIM_FREIGHT_COST_UPLIFT = 0.8

if not DB_PATH.exists():
    raise FileNotFoundError(f'Missing database: {DB_PATH}')

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
conn.row_factory = sqlite3.Row
print(f'Connected to: {DB_PATH}')
print('seed:', SEED)


## Sample Definition

- 用户口径：`customer_unique_id`
- 样本：首个有效订单，且至少留出 `60d` 观察窗口
- 主指标：`30d repeat`
- 过程指标：`late share`、`4d+ late share`、`bad review share`
- 护栏：`freight_value / order` 作为物流成本 proxy

分层随机：按 `region x first_paid_value_bucket` 做 50/50 平衡分流。

In [ ]:
FIRST_ORDER_SQL = '''
WITH pay AS (
  SELECT order_id, SUM(payment_value) AS paid_value
  FROM order_payments
  GROUP BY 1
),
itm AS (
  SELECT
    order_id,
    SUM(freight_value) AS freight_value,
    COUNT(*) AS item_cnt
  FROM order_items
  GROUP BY 1
),
review_dedup AS (
  SELECT
    order_id,
    review_score,
    ROW_NUMBER() OVER (
      PARTITION BY order_id
      ORDER BY review_answer_timestamp DESC, review_creation_date DESC, review_id DESC
    ) AS rn
  FROM order_reviews
),
review_final AS (
  SELECT order_id, review_score
  FROM review_dedup
  WHERE rn = 1
),
valid_orders AS (
  SELECT
    o.order_id,
    c.customer_unique_id,
    c.customer_state,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    COALESCE(pay.paid_value, 0) AS paid_value,
    COALESCE(itm.freight_value, 0) AS freight_value,
    COALESCE(itm.item_cnt, 0) AS item_cnt,
    CASE
      WHEN o.order_estimated_delivery_date IS NOT NULL
       AND o.order_delivered_customer_date IS NOT NULL
      THEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date)
      ELSE NULL
    END AS delay_days
  FROM orders o
  LEFT JOIN customers c
    ON o.customer_id = c.customer_id
  LEFT JOIN pay
    ON o.order_id = pay.order_id
  LEFT JOIN itm
    ON o.order_id = itm.order_id
  WHERE o.order_status NOT IN ('canceled', 'unavailable')
    AND COALESCE(pay.paid_value, 0) > 0
    AND c.customer_unique_id IS NOT NULL
    AND o.order_purchase_timestamp IS NOT NULL
),
ordered AS (
  SELECT
    vo.*,
    rf.review_score,
    CASE WHEN rf.review_score IS NOT NULL AND rf.review_score <= 2 THEN 1 ELSE 0 END AS is_bad_review,
    LEAD(vo.order_purchase_timestamp) OVER (
      PARTITION BY vo.customer_unique_id
      ORDER BY vo.order_purchase_timestamp, vo.order_id
    ) AS next_purchase_ts,
    ROW_NUMBER() OVER (
      PARTITION BY vo.customer_unique_id
      ORDER BY vo.order_purchase_timestamp, vo.order_id
    ) AS order_rank,
    MAX(vo.order_purchase_timestamp) OVER () AS max_purchase_ts
  FROM valid_orders vo
  LEFT JOIN review_final rf
    ON vo.order_id = rf.order_id
)
SELECT
  customer_unique_id,
  customer_state,
  order_id,
  order_purchase_timestamp,
  paid_value,
  freight_value,
  item_cnt,
  delay_days,
  review_score,
  is_bad_review,
  CASE WHEN delay_days > 0 THEN 1 ELSE 0 END AS is_late,
  CASE WHEN delay_days > 4 THEN 1 ELSE 0 END AS is_late_4d,
  CASE
    WHEN next_purchase_ts IS NOT NULL
     AND julianday(next_purchase_ts) <= julianday(order_purchase_timestamp) + 30
    THEN 1 ELSE 0 END AS repeat_30d
FROM ordered
WHERE order_rank = 1
  AND julianday(max_purchase_ts) - julianday(order_purchase_timestamp) >= 60
'''

rows = [dict(row) for row in conn.execute(FIRST_ORDER_SQL).fetchall()]
print('eligible first-order users:', len(rows))
print('repeat_30d baseline:', round(sum(row['repeat_30d'] for row in rows) / len(rows), 4))
print('late_share baseline:', round(sum(row['is_late'] for row in rows) / len(rows), 4))
print('late_4d_share baseline:', round(sum(row['is_late_4d'] for row in rows) / len(rows), 4))
print('bad_review_share baseline:', round(sum(row['is_bad_review'] for row in rows) / len(rows), 4))


In [ ]:
STATE_TO_REGION = {
    'RO': 'North', 'AC': 'North', 'AM': 'North', 'RR': 'North', 'PA': 'North', 'AP': 'North', 'TO': 'North',
    'MA': 'Northeast', 'PI': 'Northeast', 'CE': 'Northeast', 'RN': 'Northeast', 'PB': 'Northeast', 'PE': 'Northeast', 'AL': 'Northeast', 'SE': 'Northeast', 'BA': 'Northeast',
    'MT': 'Center-West', 'MS': 'Center-West', 'GO': 'Center-West', 'DF': 'Center-West',
    'SP': 'Southeast', 'RJ': 'Southeast', 'ES': 'Southeast', 'MG': 'Southeast',
    'PR': 'South', 'SC': 'South', 'RS': 'South',
}


def basket_value_bucket(paid_value):
    if paid_value is None:
        return 'missing'
    if paid_value < 100:
        return '<100'
    if paid_value < 200:
        return '100-200'
    return '200+'


def delay_bucket(delay_days):
    if delay_days is None:
        return 'missing'
    if delay_days <= 0:
        return 'on_time'
    if delay_days <= 3:
        return '1_3_late'
    if delay_days <= 7:
        return '4_7_late'
    return '7p_late'


def u01(key):
    digest = hashlib.sha256(key.encode()).hexdigest()[:16]
    return int(digest, 16) / float(16 ** 16)


def smooth_rate(success, total, prior, strength=50):
    return (success + prior * strength) / (total + strength)


def proportion_test(control_values, treatment_values):
    n_c = len(control_values)
    n_t = len(treatment_values)
    p_c = sum(control_values) / n_c
    p_t = sum(treatment_values) / n_t
    diff = p_t - p_c
    se = math.sqrt(p_c * (1 - p_c) / n_c + p_t * (1 - p_t) / n_t)
    z = diff / se if se else 0.0
    p_value = 2 * (1 - NormalDist().cdf(abs(z)))
    ci_low = diff - 1.96 * se
    ci_high = diff + 1.96 * se
    return {
        'control': p_c,
        'treatment': p_t,
        'diff': diff,
        'ci_low': ci_low,
        'ci_high': ci_high,
        'p_value': p_value,
        'n_control': n_c,
        'n_treatment': n_t,
    }


def mean_test(control_values, treatment_values):
    n_c = len(control_values)
    n_t = len(treatment_values)
    mean_c = sum(control_values) / n_c
    mean_t = sum(treatment_values) / n_t
    var_c = sum((x - mean_c) ** 2 for x in control_values) / (n_c - 1)
    var_t = sum((x - mean_t) ** 2 for x in treatment_values) / (n_t - 1)
    diff = mean_t - mean_c
    se = math.sqrt(var_c / n_c + var_t / n_t)
    z = diff / se if se else 0.0
    p_value = 2 * (1 - NormalDist().cdf(abs(z)))
    ci_low = diff - 1.96 * se
    ci_high = diff + 1.96 * se
    return {
        'control': mean_c,
        'treatment': mean_t,
        'diff': diff,
        'ci_low': ci_low,
        'ci_high': ci_high,
        'p_value': p_value,
        'n_control': n_c,
        'n_treatment': n_t,
    }


## Simulation Assumptions

默认场景用一个“更快但更贵”的履约 uplift：

- treatment 对正向延迟订单减少 `1.5` 天 delay
- bad review 不直接手写 uplift，而是根据新的 `delay bucket` 用历史经验率重采样
- `30d repeat` 在新的体验状态上，再额外加 `+0.35pp` 绝对 uplift
- `freight_value / order` 增加 `0.8`，作为物流成本 proxy


In [ ]:
strata = defaultdict(list)
for row in rows:
    row['region'] = STATE_TO_REGION.get(row['customer_state'], 'Other')
    row['basket_bucket'] = basket_value_bucket(row['paid_value'])
    row['delay_bucket'] = delay_bucket(row['delay_days'])
    strata[(row['region'], row['basket_bucket'])].append(row)

assigned = []
for key, items in strata.items():
    items.sort(key=lambda row: hashlib.sha256((row['customer_unique_id'] + '|' + SEED).encode()).hexdigest())
    for idx, row in enumerate(items):
        row['group'] = 'treatment' if idx % 2 else 'control'
        assigned.append(row)

rows = assigned
print('control users:', sum(1 for row in rows if row['group'] == 'control'))
print('treatment users:', sum(1 for row in rows if row['group'] == 'treatment'))

bad_review_counts = defaultdict(int)
delay_bucket_counts = defaultdict(int)
repeat_counts = defaultdict(int)
repeat_sums = defaultdict(int)

for row in rows:
    delay_bucket_counts[row['delay_bucket']] += 1
    bad_review_counts[row['delay_bucket']] += row['is_bad_review']
    repeat_key = (row['is_bad_review'], row['delay_bucket'])
    repeat_counts[repeat_key] += 1
    repeat_sums[repeat_key] += row['repeat_30d']

global_bad_review_rate = sum(row['is_bad_review'] for row in rows) / len(rows)
global_repeat_rate = sum(row['repeat_30d'] for row in rows) / len(rows)

bad_review_rate_by_delay = {
    bucket: smooth_rate(bad_review_counts[bucket], delay_bucket_counts[bucket], global_bad_review_rate)
    for bucket in delay_bucket_counts
}

repeat_rate_lookup = {
    key: smooth_rate(repeat_sums[key], repeat_counts[key], global_repeat_rate)
    for key in repeat_counts
}

for row in rows:
    if row['group'] == 'treatment':
        original_delay = row['delay_days']
        if original_delay is None:
            sim_delay = None
        elif original_delay > 0:
            sim_delay = original_delay - SIM_DELAY_REDUCTION_DAYS
        else:
            sim_delay = original_delay

        sim_delay_bucket = delay_bucket(sim_delay)
        sim_bad_review_prob = bad_review_rate_by_delay[sim_delay_bucket]
        sim_bad_review = 1 if u01(row['customer_unique_id'] + '|bad_review') < sim_bad_review_prob else 0

        base_repeat_rate = repeat_rate_lookup[(sim_bad_review, sim_delay_bucket)]
        sim_repeat_rate = min(1.0, max(0.0, base_repeat_rate + SIM_REPEAT_UPLIFT_ABS))
        sim_repeat_30d = 1 if u01(row['customer_unique_id'] + '|repeat_30d') < sim_repeat_rate else 0

        row['sim_delay_days'] = sim_delay
        row['sim_is_late'] = 1 if sim_delay is not None and sim_delay > 0 else 0
        row['sim_is_late_4d'] = 1 if sim_delay is not None and sim_delay > 4 else 0
        row['sim_bad_review'] = sim_bad_review
        row['sim_repeat_30d'] = sim_repeat_30d
        row['sim_freight_value'] = (row['freight_value'] or 0.0) + SIM_FREIGHT_COST_UPLIFT
    else:
        row['sim_delay_days'] = row['delay_days']
        row['sim_is_late'] = row['is_late']
        row['sim_is_late_4d'] = row['is_late_4d']
        row['sim_bad_review'] = row['is_bad_review']
        row['sim_repeat_30d'] = row['repeat_30d']
        row['sim_freight_value'] = row['freight_value'] or 0.0

balance_paid = mean_test(
    [row['paid_value'] for row in rows if row['group'] == 'control'],
    [row['paid_value'] for row in rows if row['group'] == 'treatment'],
)
balance_late = proportion_test(
    [row['is_late'] for row in rows if row['group'] == 'control'],
    [row['is_late'] for row in rows if row['group'] == 'treatment'],
)
print('balance check - paid_value diff:', round(balance_paid['diff'], 4), 'p=', round(balance_paid['p_value'], 4))
print('balance check - late_share diff:', round(balance_late['diff'], 4), 'p=', round(balance_late['p_value'], 4))


In [ ]:
results = {
    'repeat_30d': proportion_test(
        [row['sim_repeat_30d'] for row in rows if row['group'] == 'control'],
        [row['sim_repeat_30d'] for row in rows if row['group'] == 'treatment'],
    ),
    'late_share': proportion_test(
        [row['sim_is_late'] for row in rows if row['group'] == 'control'],
        [row['sim_is_late'] for row in rows if row['group'] == 'treatment'],
    ),
    'late_4d_share': proportion_test(
        [row['sim_is_late_4d'] for row in rows if row['group'] == 'control'],
        [row['sim_is_late_4d'] for row in rows if row['group'] == 'treatment'],
    ),
    'bad_review_share': proportion_test(
        [row['sim_bad_review'] for row in rows if row['group'] == 'control'],
        [row['sim_bad_review'] for row in rows if row['group'] == 'treatment'],
    ),
    'freight_value_per_order': mean_test(
        [row['sim_freight_value'] for row in rows if row['group'] == 'control'],
        [row['sim_freight_value'] for row in rows if row['group'] == 'treatment'],
    ),
}

for metric, summary in results.items():
    print(metric)
    print(f"  control={summary['control']:.4%}" if metric != 'freight_value_per_order' else f"  control={summary['control']:.2f}")
    print(f"  treatment={summary['treatment']:.4%}" if metric != 'freight_value_per_order' else f"  treatment={summary['treatment']:.2f}")
    print(f"  diff={summary['diff']:.4%}" if metric != 'freight_value_per_order' else f"  diff={summary['diff']:.2f}")
    print(
        f"  95% CI=({summary['ci_low']:.4%}, {summary['ci_high']:.4%})"
        if metric != 'freight_value_per_order'
        else f"  95% CI=({summary['ci_low']:.2f}, {summary['ci_high']:.2f})"
    )
    print(f"  p_value={summary['p_value']:.6f}")
    print()

launch_recommendation = 'continue_running'
if (
    results['repeat_30d']['p_value'] < 0.05
    and results['late_share']['diff'] < 0
    and results['bad_review_share']['diff'] <= 0
    and results['freight_value_per_order']['diff'] <= 0
):
    launch_recommendation = 'launch'
elif results['late_share']['p_value'] >= 0.05 and results['repeat_30d']['p_value'] >= 0.05:
    launch_recommendation = 'do_not_launch'

print('decision:', launch_recommendation)
print('interpretation:')
if launch_recommendation == 'launch':
    print('- Main and process metrics improved, and guardrails did not worsen.')
elif launch_recommendation == 'continue_running':
    print('- Main and process metrics improved, but the cost proxy worsened. Keep holdout and continue monitoring before full rollout.')
else:
    print('- The simulated uplift is not strong enough or guardrails worsened. Do not launch yet.')
